In [4]:
import onnxruntime as ort
import numpy as np
import scipy.special
from PIL import Image
# 预处理图像
def preprocess_image(image, resize_size=256, crop_size=224, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    image = image.resize((resize_size, resize_size), Image.BILINEAR)
    w, h = image.size
    left = (w - crop_size) / 2
    top = (h - crop_size) / 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    image = np.array(image).astype(np.float32)
    image = image / 255.0
    image = (image - mean) / std
    image = np.transpose(image, (2, 0, 1))
    image = image.reshape((1,) + image.shape)
    return image


session = ort.InferenceSession('flower-detection.onnx')

with open('labels.txt') as f:
    labels = [line.strip() for line in f.readlines()]

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

image = Image.open('flower_test.png').convert('RGB')
processed_image = preprocess_image(image)

processed_image = processed_image.astype(np.float32)

output = session.run([output_name], {input_name: processed_image})[0]

accuracy = scipy.special.softmax(output, axis=-1)

print(accuracy)

predicted_idx = np.argmax(accuracy)

prob_percentage = accuracy[0, predicted_idx] * 100
print(predicted_idx)

predicted_label = labels[predicted_idx]
# 输出预测结果，包含百分比形式的概率
print(f"Predicted class: {predicted_label}, Accuracy: {prob_percentage:.2f}%")



[[2.39585916e-06 2.93439170e-05 8.90261376e-08 2.50165595e-08
  2.15281702e-07 7.72698684e-07 5.03510691e-08 4.33364721e-06
  3.11128633e-06 2.87395750e-07 2.06182049e-05 4.62672433e-05
  7.48300999e-06 1.99655506e-06 1.14799455e-04 7.97592202e-06
  2.71746339e-05 5.49619244e-06 5.08169296e-06 7.89827573e-06
  8.94737605e-06 1.51143286e-06 6.13679561e-07 1.63025732e-06
  2.02331921e-06 7.95757296e-06 2.86248287e-05 1.29586379e-05
  2.31118702e-05 5.93643381e-06 5.85330054e-06 3.12638549e-05
  1.10922520e-05 1.14902235e-07 6.18382956e-07 1.51623033e-06
  1.20154400e-05 2.36067058e-06 1.56659837e-06 1.53665076e-06
  1.54121026e-05 3.22093524e-06 3.11923941e-06 1.86572618e-06
  3.47824721e-06 7.89826402e-07 1.05795280e-05 6.15833778e-06
  1.29052978e-07 1.69857884e-07 3.51939320e-07 2.16287006e-07
  1.41029777e-05 2.08492693e-05 1.76255685e-06 4.61562104e-06
  3.04144032e-06 3.70713951e-06 4.65100948e-06 3.42443454e-05
  4.25206463e-06 5.64949289e-07 1.45854756e-06 1.19760125e-06
  2.3803